# 02_silver
Builds the silver layer from the latest bronze batch:
- `silver.pageviews_daily`: one row per article per day; gap fill (gaps under 3 days → 0, longer → NULL) and outlier flag (5x trailing 56-day median). Loaded with MERGE.
- `silver.pageviews_weekly`: complete Monday-start weeks, index 2019 = 100. Rebuilt each run.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW latest_pageviews AS
SELECT lang, topic, input_title AS article, batch_id, response_body
FROM (
  SELECT *, row_number() OVER (PARTITION BY lang, topic ORDER BY ingested_at DESC) AS rn
  FROM nba_demand.bronze.api_responses
  WHERE source = 'wikimedia_pageviews' AND http_status = 200
)
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW parsed AS
SELECT lang, topic, article, batch_id,
       to_date(substr(item.timestamp, 1, 8), 'yyyyMMdd') AS view_date,
       item.views AS views
FROM (
  SELECT lang, topic, article, batch_id,
         explode(from_json(response_body, 'items ARRAY<STRUCT<timestamp: STRING, views: BIGINT>>').items) AS item
  FROM latest_pageviews
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW calendar AS
SELECT lang, topic, article, batch_id, explode(sequence(first_day, last_day, INTERVAL 1 DAY)) AS view_date
FROM (
  SELECT lang, topic, article, batch_id, min(view_date) AS first_day, max(view_date) AS last_day
  FROM parsed
  GROUP BY lang, topic, article, batch_id
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gap_filled AS
SELECT lang, topic, article, batch_id, view_date, views_reported,
       CASE WHEN NOT is_missing THEN views_reported
            WHEN run_days < 3 THEN 0 END AS views,
       is_missing AND run_days < 3 AS was_gap_filled
FROM (
  SELECT *, count(*) OVER (PARTITION BY lang, topic, is_missing, run_id) AS run_days
  FROM (
    SELECT c.lang, c.topic, c.article, c.batch_id, c.view_date,
           p.views AS views_reported,
           p.views IS NULL AS is_missing,
           datediff(c.view_date, DATE'1970-01-01')
             - row_number() OVER (PARTITION BY c.lang, c.topic, p.views IS NULL ORDER BY c.view_date) AS run_id
    FROM calendar c
    LEFT JOIN parsed p ON p.lang = c.lang AND p.topic = c.topic AND p.view_date = c.view_date
  )
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_new AS
SELECT lang, topic, article, view_date, views_reported, views, was_gap_filled,
       CASE WHEN views > 5 * trailing_median_56d AND trailing_median_56d > 0
            THEN 5 * trailing_median_56d ELSE views END AS views_capped,
       coalesce(views > 5 * trailing_median_56d AND trailing_median_56d > 0, false) AS was_capped,
       trailing_median_56d,
       batch_id AS source_batch_id
FROM (
  SELECT *,
         percentile(views, 0.5) OVER (PARTITION BY lang, topic ORDER BY view_date
                                      ROWS BETWEEN 56 PRECEDING AND 1 PRECEDING) AS trailing_median_56d
  FROM gap_filled
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nba_demand.silver.pageviews_daily (
  lang                STRING    COMMENT 'Wikipedia language edition',
  topic               STRING    COMMENT 'basketball or nba',
  article             STRING    COMMENT 'Canonical article title',
  view_date           DATE      COMMENT 'Calendar day (UTC)',
  views_reported      BIGINT    COMMENT 'Views as returned by the API; NULL if the API returned no row for this day',
  views               BIGINT    COMMENT 'Views after gap fill: gaps under 3 days set to 0, longer gaps left NULL',
  was_gap_filled      BOOLEAN   COMMENT 'True if views was set to 0 by the gap fill',
  views_capped        DOUBLE    COMMENT 'views capped at 5x the trailing 56-day median (outlier flag, not used by the forecast)',
  was_capped          BOOLEAN   COMMENT 'True if views exceeded the cap',
  trailing_median_56d DOUBLE    COMMENT 'Median of views over the previous 56 days',
  source_batch_id     STRING    COMMENT 'Bronze batch that last changed this row',
  updated_at          TIMESTAMP COMMENT 'When this row was last inserted or updated'
)
COMMENT 'Silver: one row per article per day, cleaned. Loaded with MERGE from the latest bronze batch.';

MERGE INTO nba_demand.silver.pageviews_daily AS t
USING daily_new AS s
ON t.lang = s.lang AND t.topic = s.topic AND t.view_date = s.view_date
WHEN MATCHED AND NOT (t.views_reported <=> s.views_reported AND t.views <=> s.views
                      AND t.views_capped <=> s.views_capped AND t.article <=> s.article) THEN
  UPDATE SET article = s.article, views_reported = s.views_reported, views = s.views,
             was_gap_filled = s.was_gap_filled, views_capped = s.views_capped, was_capped = s.was_capped,
             trailing_median_56d = s.trailing_median_56d, source_batch_id = s.source_batch_id,
             updated_at = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (lang, topic, article, view_date, views_reported, views, was_gap_filled, views_capped,
          was_capped, trailing_median_56d, source_batch_id, updated_at)
  VALUES (s.lang, s.topic, s.article, s.view_date, s.views_reported, s.views, s.was_gap_filled, s.views_capped,
          s.was_capped, s.trailing_median_56d, s.source_batch_id, current_timestamp());

In [0]:
%sql
CREATE OR REPLACE TABLE nba_demand.silver.pageviews_weekly
COMMENT 'Silver: one row per article per Monday-start week (complete 7-day weeks only), with a 2019 = 100 index.'
AS
WITH weekly AS (
  SELECT lang, topic, article,
         CAST(date_trunc('WEEK', view_date) AS DATE) AS week_start,
         min(extract(YEAROFWEEK FROM view_date)) AS iso_year,
         min(weekofyear(view_date)) AS iso_week,
         sum(views) AS views,
         sum(views_capped) AS views_capped,
         count(views) AS days_observed,
         count_if(was_capped) AS days_capped
  FROM nba_demand.silver.pageviews_daily
  GROUP BY lang, topic, article, CAST(date_trunc('WEEK', view_date) AS DATE)
  HAVING count(*) = 7
),
baseline_2019 AS (
  SELECT lang, topic, avg(views) AS avg_views_2019, avg(views_capped) AS avg_views_capped_2019
  FROM weekly
  WHERE iso_year = 2019 AND days_observed = 7
  GROUP BY lang, topic
)
SELECT w.lang, w.topic, w.article, w.week_start, w.iso_year, w.iso_week,
       w.views, w.views_capped, w.days_observed, w.days_capped,
       w.days_observed < 7 AS is_incomplete,
       round(w.views / b.avg_views_2019 * 100, 2) AS index_2019,
       round(w.views_capped / b.avg_views_capped_2019 * 100, 2) AS index_2019_capped
FROM weekly w
JOIN baseline_2019 b ON b.lang = w.lang AND b.topic = w.topic;

In [0]:
%sql
SELECT topic, count(*) AS days, min(view_date) AS first_day, max(view_date) AS last_day,
       count_if(views IS NULL) AS days_null, count_if(was_gap_filled) AS days_gap_filled,
       count_if(was_capped) AS days_capped
FROM nba_demand.silver.pageviews_daily
GROUP BY topic;

In [0]:
%sql
SELECT topic, count(*) AS weeks, min(week_start) AS first_week, max(week_start) AS last_week,
       count_if(is_incomplete) AS incomplete_weeks,
       round(avg(CASE WHEN iso_year = 2019 THEN index_2019 END), 1) AS avg_index_2019
FROM nba_demand.silver.pageviews_weekly
GROUP BY topic;